# Spell Feature Engineering

This notebook extracts features from D&D 5e spell data.

**Input:** `data/spells.csv`
**Output:** Updated `data/spells.csv` with additional feature columns

## Imports and Setup

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Detect execution context
cwd = Path.cwd()
if cwd.name == 'notebooks':
    DATA_DIR = '../data'
    sys.path.insert(0, '.')
    from helper_files import parse_spell_targets, calculate_average_damage
else:
    DATA_DIR = './data'
    sys.path.insert(0, '.')
    from notebooks.helper_files import parse_spell_targets, calculate_average_damage

print(f"Data directory: {DATA_DIR}")
print("Imports successful")

Data directory: ./data
Imports successful


## Configuration

Tunable parameters for DPR calculations. Adjust these to observe different model behaviors.

In [2]:
# =============================================================================
# DPR CALCULATION PARAMETERS
# =============================================================================
# Adjust these values to tune how spell DPR is calculated

# AOE_PASSTHROUGH_MULTIPLIER: Scales the damage for AoE spells
# - 1.0 = full damage passes through (default, no modification)
# - 0.5 = AoE damage counts as 50% effective (e.g., accounting for saves)
# - Use this to model the "effective" damage of AoE vs single-target
AOE_PASSTHROUGH_MULTIPLIER = 1.0

# TARGET_CAP: Maximum number of targets used in modified_dpr calculation
# - None = no cap, use full estimated_targets (default)
# - 4 = cap at 4 targets (common design assumption for balanced encounters)
# - 2 = very conservative cap
# Example: Fireball estimates 7.5 targets, but with TARGET_CAP=4, uses 4
TARGET_CAP = 4

print(f"DPR Configuration:")
print(f"  AOE_PASSTHROUGH_MULTIPLIER = {AOE_PASSTHROUGH_MULTIPLIER}")
print(f"  TARGET_CAP = {TARGET_CAP if TARGET_CAP else 'None (no cap)'}")

DPR Configuration:
  AOE_PASSTHROUGH_MULTIPLIER = 1.0
  TARGET_CAP = 4


In [3]:
def calculate_modified_dpr(row, aoe_multiplier=AOE_PASSTHROUGH_MULTIPLIER, target_cap=TARGET_CAP):
    """
    Calculate modified DPR for a spell based on base damage and targets.
    
    Parameters:
    -----------
    row : pd.Series
        Row from spell dataframe with 'avg_damage', 'is_aoe', 'estimated_targets', 'target_count'
    aoe_multiplier : float
        Multiplier applied to AoE spell damage (default from config)
    target_cap : int or None
        Maximum number of targets to count (default from config)
    
    Returns:
    --------
    float : modified_dpr value
    
    Formula:
    --------
    For AoE spells:
        effective_targets = min(estimated_targets, target_cap) if target_cap else estimated_targets
        modified_dpr = base_dpr × effective_targets × aoe_multiplier
    
    For non-AoE spells:
        modified_dpr = base_dpr × target_count (no multiplier applied)
    """
    base_dpr = row['avg_damage'] if pd.notna(row['avg_damage']) else 0
    
    if base_dpr == 0:
        return 0.0
    
    if row['is_aoe']:
        # AoE spell: use estimated_targets with cap and multiplier
        targets = row['estimated_targets'] if pd.notna(row['estimated_targets']) else 1
        if target_cap is not None:
            targets = min(targets, target_cap)
        return base_dpr * targets * aoe_multiplier
    else:
        # Non-AoE spell: use target_count directly (no multiplier)
        targets = row['target_count'] if pd.notna(row['target_count']) else 1
        if target_cap is not None:
            targets = min(targets, target_cap)
        return base_dpr * targets

print("DPR calculation function defined")

DPR calculation function defined


## Load Spell Data

In [4]:
load_path = DATA_DIR + '/spells.csv'
df = pd.read_csv(load_path)

In [5]:
# Parse target information for each spell
target_data = df['description'].apply(parse_spell_targets)

# Extract individual columns from the dict results
df['target_count'] = target_data.apply(lambda x: x['target_count'])
df['is_aoe'] = target_data.apply(lambda x: x['is_aoe'])
df['aoe_type'] = target_data.apply(lambda x: x['aoe_type'])
df['aoe_size'] = target_data.apply(lambda x: x['aoe_size'])
df['estimated_targets'] = target_data.apply(lambda x: x['estimated_targets'])

# For non-AoE spells with target_count, use that as estimated_targets
df.loc[df['target_count'].notna() & df['estimated_targets'].isna(), 'estimated_targets'] = df['target_count']

print("Target features extracted")

Target features extracted


## Calculate DPR Columns

Using the configurable parameters to calculate:
- `base_dpr`: Raw average damage (same as avg_damage)
- `effective_targets`: Targets used in calculation (respects TARGET_CAP)
- `modified_dpr`: base_dpr × effective_targets × AOE_PASSTHROUGH_MULTIPLIER (for AoE)

In [6]:
# Calculate DPR columns
df['base_dpr'] = df['avg_damage'].fillna(0)

# Calculate effective targets (respects TARGET_CAP)
def get_effective_targets(row):
    if row['is_aoe']:
        targets = row['estimated_targets'] if pd.notna(row['estimated_targets']) else 1
    else:
        targets = row['target_count'] if pd.notna(row['target_count']) else 1
    
    if TARGET_CAP is not None:
        targets = min(targets, TARGET_CAP)
    return targets

df['effective_targets'] = df.apply(get_effective_targets, axis=1)

# Calculate modified DPR
df['modified_dpr'] = df.apply(calculate_modified_dpr, axis=1)

# Show example calculations
print("=== DPR Calculation Examples ===")
print(f"Config: AOE_PASSTHROUGH_MULTIPLIER={AOE_PASSTHROUGH_MULTIPLIER}, TARGET_CAP={TARGET_CAP}")
print()

examples = ['Fireball', 'Magic Missile', 'Cone of Cold', 'Scorching Ray', 'Eldritch Blast']
for spell_name in examples:
    spell = df[df['spell_name'] == spell_name]
    if len(spell) > 0:
        row = spell.iloc[0]
        aoe_marker = "(AoE)" if row['is_aoe'] else ""
        print(f"{spell_name} {aoe_marker}:")
        print(f"  base_dpr={row['base_dpr']:.1f}, effective_targets={row['effective_targets']:.1f}, modified_dpr={row['modified_dpr']:.1f}")

=== DPR Calculation Examples ===
Config: AOE_PASSTHROUGH_MULTIPLIER=1.0, TARGET_CAP=4

Fireball (AoE):
  base_dpr=28.0, effective_targets=4.0, modified_dpr=112.0
Magic Missile :
  base_dpr=3.5, effective_targets=3.0, modified_dpr=10.5
Cone of Cold (AoE):
  base_dpr=36.0, effective_targets=4.0, modified_dpr=144.0
Scorching Ray :
  base_dpr=7.0, effective_targets=3.0, modified_dpr=21.0
Eldritch Blast :
  base_dpr=5.5, effective_targets=1.0, modified_dpr=5.5


## Extract Condition Features

Parse spell descriptions to identify which conditions they can inflict.
Mirrors the monster features: `inflicts_blinded`, `inflicts_charmed`, etc.

In [7]:
import re

# =============================================================================
# CONDITION FEATURES (matches monster feature naming)
# =============================================================================
# List of conditions to detect (same as 1_feature_engineering.ipynb)
CONDITIONS = [
    'blinded', 'charmed', 'deafened', 'frightened', 'incapacitated',
    'paralyzed', 'petrified', 'poisoned', 'prone', 'restrained', 'stunned'
]

def extract_condition_features(description):
    """
    Extract condition infliction features from spell description.
    Returns dict with inflicts_{condition} boolean flags.
    """
    if pd.isna(description):
        return {f'inflicts_{c}': False for c in CONDITIONS}
    
    desc_lower = description.lower()
    result = {}
    
    for condition in CONDITIONS:
        # Patterns that indicate the spell inflicts this condition:
        # - "target is [condition]"
        # - "becomes [condition]"
        # - "be [condition]"
        # - "creature is [condition]"
        # - "knocked prone" / "falls prone"
        # - "[condition] for the duration"
        
        patterns = [
            rf'\b{condition}\b',  # Basic presence of the condition word
        ]
        
        # Check if any pattern matches
        found = any(re.search(p, desc_lower) for p in patterns)
        
        # Exclude false positives: spells that REMOVE or PREVENT conditions
        # e.g., "can't be charmed", "immunity to being frightened"
        if found:
            exclusion_patterns = [
                rf"can't be {condition}",
                rf"cannot be {condition}",
                rf"immune to .+{condition}",
                rf"immunity to .+{condition}",
                rf"ends? .+{condition}",
                rf"cured? of .+{condition}",
                rf"no longer {condition}",
            ]
            if any(re.search(p, desc_lower) for p in exclusion_patterns):
                found = False
        
        result[f'inflicts_{condition}'] = found
    
    return result

# Apply to all spells
condition_features = df['description'].apply(extract_condition_features).apply(pd.Series)
already_joined = False
for col in condition_features.columns:
    if col in df.columns:
        already_joined = True
if already_joined == False:
    df = pd.concat([df, condition_features], axis=1)

# Summary
print("=== Condition Features ===")
for condition in CONDITIONS:
    col = f'inflicts_{condition}'
    count = df[col].sum()
    if count > 0:
        print(f"  {col}: {count} spells")

# Show examples
print("\n=== Example Spells with Conditions ===")
condition_examples = [
    ('Hold Person', 'paralyzed'),
    ('Fear', 'frightened'),
    ('Blindness/Deafness', 'blinded'),
    ('Tasha\'s Hideous Laughter', 'prone'),
    ('Command', 'prone'),
]
for spell_name, expected_condition in condition_examples:
    spell = df[df['spell_name'] == spell_name]
    if len(spell) > 0:
        row = spell.iloc[0]
        col = f'inflicts_{expected_condition}'
        print(f"  {spell_name}: {col}={row[col]}")

=== Condition Features ===
  inflicts_blinded: 17 spells
  inflicts_charmed: 22 spells
  inflicts_deafened: 6 spells
  inflicts_frightened: 11 spells
  inflicts_incapacitated: 12 spells
  inflicts_paralyzed: 5 spells
  inflicts_poisoned: 4 spells
  inflicts_prone: 12 spells
  inflicts_restrained: 9 spells
  inflicts_stunned: 6 spells

=== Example Spells with Conditions ===
  Hold Person: inflicts_paralyzed=True
  Fear: inflicts_frightened=True
  Blindness/Deafness: inflicts_blinded=True
  Tasha's Hideous Laughter: inflicts_prone=True
  Command: inflicts_prone=False


## Extract Buff/Debuff Features

Parse spell descriptions to identify:
- `grants_flying`: Spells that grant flying speed (Fly, Levitate, etc.)
- `ac_bonus`: AC bonus granted (Shield = +5, Shield of Faith = +2)
- `attack_bonus`: Attack roll bonus (Bless = avg +2.5)
- `save_bonus`: Saving throw bonus
- `grants_advantage`: Grants advantage on attacks/saves
- `inflicts_disadvantage`: Inflicts disadvantage on target

In [8]:
# =============================================================================
# BUFF/DEBUFF FEATURES
# =============================================================================

def extract_buff_debuff_features(description):
    """
    Extract buff/debuff features from spell description.
    Returns dict with flying, AC bonus, attack bonus, advantage/disadvantage.
    """
    if pd.isna(description):
        return {
            'grants_flying': False,
            'ac_bonus': 0,
            'attack_bonus': 0.0,
            'save_bonus': 0.0,
            'grants_advantage': False,
            'inflicts_disadvantage': False
        }
    
    desc_lower = description.lower()
    result = {}
    
    # GRANTS FLYING
    flying_patterns = [
        r'flying speed',
        r'gains? a flying',
        r'target can fly',
        r'creature can fly',
        r'you can fly',
        r'ability to fly',
    ]
    result['grants_flying'] = any(re.search(p, desc_lower) for p in flying_patterns)
    
    # AC BONUS
    ac_bonus = 0
    ac_match = re.search(r'\+(\d+)\s+bonus to ac', desc_lower)
    if ac_match:
        ac_bonus = int(ac_match.group(1))
    mage_armor_match = re.search(r'ac becomes (\d+)\s*\+', desc_lower)
    if mage_armor_match and ac_bonus == 0:
        base_ac = int(mage_armor_match.group(1))
        ac_bonus = max(0, base_ac - 10)
    result['ac_bonus'] = ac_bonus
    
    # ATTACK BONUS
    attack_bonus = 0.0
    if re.search(r'd4.+add.+(?:to the )?attack roll', desc_lower):
        attack_bonus = 2.5
    attack_match = re.search(r'\+(\d+)\s+(?:bonus\s+)?to\s+attack\s+rolls?', desc_lower)
    if attack_match:
        attack_bonus = float(attack_match.group(1))
    result['attack_bonus'] = attack_bonus
    
    # SAVE BONUS
    save_bonus = 0.0
    if re.search(r'd4.+add.+saving throw', desc_lower):
        save_bonus = 2.5
    save_match = re.search(r'\+(\d+)\s+(?:bonus\s+)?to\s+saving\s+throws?', desc_lower)
    if save_match:
        save_bonus = float(save_match.group(1))
    result['save_bonus'] = save_bonus
    
    # GRANTS ADVANTAGE
    advantage_patterns = [
        r'(?:has|have|gains?)\s+advantage\s+on\s+(?:attack|weapon)',
        r'attack rolls?.+have advantage',
        r'advantage on.+attack rolls?',
        r'attack roll.+has advantage',
    ]
    result['grants_advantage'] = any(re.search(p, desc_lower) for p in advantage_patterns)
    
    # INFLICTS DISADVANTAGE
    disadvantage_patterns = [
        r'(?:has|have)\s+disadvantage\s+on\s+(?:attack|weapon)',
        r'attack rolls?.+have disadvantage',
        r'disadvantage on.+attack rolls?',
        r'(?:has|have)\s+disadvantage\s+on\s+ability\s+checks',
        r'(?:has|have)\s+disadvantage\s+on\s+(?:strength|dexterity|constitution|intelligence|wisdom|charisma)',
    ]
    result['inflicts_disadvantage'] = any(re.search(p, desc_lower) for p in disadvantage_patterns)
    
    return result

# Apply to all spells (guard against re-runs)
buff_cols = ['grants_flying', 'ac_bonus', 'attack_bonus', 'save_bonus', 'grants_advantage', 'inflicts_disadvantage']
if not all(col in df.columns for col in buff_cols):
    buff_features = df['description'].apply(extract_buff_debuff_features).apply(pd.Series)
    df = pd.concat([df, buff_features], axis=1)
else:
    print("Buff/debuff columns already exist, skipping extraction")

# Summary
print("=== Buff/Debuff Features ===")
print(f"  grants_flying: {df['grants_flying'].sum()} spells")
print(f"  ac_bonus > 0: {(df['ac_bonus'] > 0).sum()} spells")
print(f"  attack_bonus > 0: {(df['attack_bonus'] > 0).sum()} spells")
print(f"  save_bonus > 0: {(df['save_bonus'] > 0).sum()} spells")
print(f"  grants_advantage: {df['grants_advantage'].sum()} spells")
print(f"  inflicts_disadvantage: {df['inflicts_disadvantage'].sum()} spells")

Buff/debuff columns already exist, skipping extraction
=== Buff/Debuff Features ===
  grants_flying: 10 spells
  ac_bonus > 0: 8 spells
  attack_bonus > 0: 3 spells
  save_bonus > 0: 3 spells
  grants_advantage: 37 spells
  inflicts_disadvantage: 32 spells


## Target Analysis

In [9]:
# Check specific spells to verify parsing
examples = [
    'Magic Missile',      # Should be 3 targets (darts)
    'Fireball',           # Should be AoE radius ~7.5 targets
    'Cure Wounds',        # Should be 1 target
    'Scorching Ray',      # Should be 3 targets (rays)
    'Lightning Bolt',     # Should be AoE line ~3 targets
    'Cone of Cold',       # Should be AoE cone ~10.8 targets
    'Hold Person',        # Should be 1 target
    'Chain Lightning',    # Should be multi-target or AoE
    'Eldritch Blast',     # Should be 1 target (at base level)
    'Burning Hands',      # Should be AoE cone (small)
]

print("=== Example Spells ===")
for spell_name in examples:
    spell = df[df['spell_name'] == spell_name]
    if len(spell) > 0:
        row = spell.iloc[0]
        if row['is_aoe']:
            print(f"{spell_name}: AoE {row['aoe_type']} ({row['aoe_size']}) → ~{row['estimated_targets']} targets")
        elif pd.notna(row['target_count']):
            print(f"{spell_name}: {int(row['target_count'])} target(s)")
        else:
            print(f"{spell_name}: unknown targeting")

=== Example Spells ===
Magic Missile: 3 target(s)
Fireball: AoE radius (20-foot) → ~7.5 targets
Cure Wounds: 1 target(s)
Scorching Ray: 3 target(s)
Lightning Bolt: AoE line (100-foot) → ~3.0 targets
Cone of Cold: AoE cone (60-foot) → ~10.8 targets
Hold Person: 1 target(s)
Chain Lightning: 3 target(s)
Eldritch Blast: 1 target(s)
Burning Hands: AoE cone (15-foot) → ~1.0 targets


In [10]:
# AoE type breakdown
print("\n=== AoE Types ===")
aoe_spells = df[df['is_aoe']]
print(aoe_spells['aoe_type'].value_counts())


=== AoE Types ===
aoe_type
radius    80
cube      35
cone      17
area      12
line       9
square     8
Name: count, dtype: int64


In [11]:
# Target count distribution (non-AoE)
print("\n=== Target Count Distribution (non-AoE) ===")
non_aoe = df[~df['is_aoe'] & df['target_count'].notna()]
print(non_aoe['target_count'].value_counts().sort_index())


=== Target Count Distribution (non-AoE) ===
target_count
1.0     316
2.0       1
3.0       8
4.0       2
5.0       4
6.0       3
8.0       4
10.0      6
Name: count, dtype: int64


## Verify Examples

In [12]:
# Check specific spells to verify parsing
examples = [
    'Magic Missile',      # Should be 3 targets (darts)
    'Fireball',           # Should be AoE radius
    'Cure Wounds',        # Should be 1 target
    'Scorching Ray',      # Should be 3 targets (rays)
    'Lightning Bolt',     # Should be AoE line
    'Cone of Cold',       # Should be AoE cone
    'Hold Person',        # Should be 1 target
    'Chain Lightning',    # Should be multi-target or AoE
    'Eldritch Blast',     # Should be 1 target (at base level)
]

print("=== Example Spells ===")
for spell_name in examples:
    spell = df[df['spell_name'] == spell_name]
    if len(spell) > 0:
        row = spell.iloc[0]
        if row['is_aoe']:
            print(f"{spell_name}: AoE ({row['aoe_type']}, {row['aoe_size']})")
        elif pd.notna(row['target_count']):
            print(f"{spell_name}: {int(row['target_count'])} target(s)")
        else:
            print(f"{spell_name}: unknown targeting")

=== Example Spells ===
Magic Missile: 3 target(s)
Fireball: AoE (radius, 20-foot)
Cure Wounds: 1 target(s)
Scorching Ray: 3 target(s)
Lightning Bolt: AoE (line, 100-foot)
Cone of Cold: AoE (cone, 60-foot)
Hold Person: 1 target(s)
Chain Lightning: 3 target(s)
Eldritch Blast: 1 target(s)


# Top single-target damage spells


In [13]:
damage_spells = df[df['avg_damage']>0].reset_index(drop=True)

In [14]:
print("\\n=== Top 10 Single-Target Damage Spells ===")
top_single = damage_spells[damage_spells['target_count'] == 1].nlargest(10, 'avg_damage')
print(top_single[['spell_name', 'level', 'damage_dice', 'avg_damage']].to_string(index=False))

\n=== Top 10 Single-Target Damage Spells ===
                    spell_name  level damage_dice  avg_damage
                   Time Ravage      9       10d12        65.0
               Finger of Death      7    7d8 + 30        61.5
                          Harm      6        14d6        49.0
            Psychic Crush (UA)      6        12d6        42.0
                 Reality Break      8        6d12        39.0
                        Blight      4         8d8        36.0
Raulothim's Psychic Lance (UA)      4        10d6        35.0
                 Blade Barrier      6        6d10        33.0
         Negative Energy Flood      5        5d12        32.5
               Banishing Smite      5        5d10        27.5


In [15]:
# Calculate total estimated damage (damage × estimated targets)
damage_spells['total_estimated_damage'] = damage_spells['avg_damage'] * damage_spells['estimated_targets']

print("\n=== Top 10 Spells by Total Estimated Damage ===")
print("(avg_damage × estimated_targets)")
top_total = damage_spells.dropna(subset=['total_estimated_damage']).nlargest(10, 'total_estimated_damage')
print(top_total[['spell_name', 'level', 'damage_dice', 'avg_damage', 'estimated_targets', 'total_estimated_damage']].to_string(index=False))


=== Top 10 Spells by Total Estimated Damage ===
(avg_damage × estimated_targets)
               spell_name  level damage_dice  avg_damage  estimated_targets  total_estimated_damage
                   Symbol      7       10d10        55.0               67.9                 3734.50
               Earthquake      8         5d6        17.5              188.5                 3298.75
                 Sunburst      8        12d6        42.0               67.9                 2851.80
       Maddening Darkness      8         8d8        36.0               67.9                 2444.40
Otiluke's Freezing Sphere      6        10d6        35.0               67.9                 2376.50
             Meteor Swarm      9        20d6        70.0               30.2                 2114.00
          Circle of Death      6         8d6        28.0               67.9                 1901.20
           Call Lightning      3        3d10        16.5               67.9                 1120.35
           Conjure

In [16]:
# Combine damage and target info for damage spells
damage_spells = df[df['avg_damage'] > 0].copy()

print(f"=== Damage Spells: {len(damage_spells)} ===")
print(f"\nAoE damage spells: {damage_spells['is_aoe'].sum()}")
print(f"Single-target damage: {(damage_spells['target_count'] == 1).sum()}")
print(f"Multi-target damage: {((damage_spells['target_count'] > 1) & ~damage_spells['is_aoe']).sum()}")

=== Damage Spells: 210 ===

AoE damage spells: 97
Single-target damage: 94
Multi-target damage: 7


In [17]:
# Final summary
print("\n=== Final Summary ===")
print(f"Total spells: {len(df)}")
print(f"With damage: {(df['avg_damage'] > 0).sum()}")
print(f"AoE: {df['is_aoe'].sum()}")
print(f"Single-target: {(df['target_count'] == 1).sum()}")
print(f"Multi-target (non-AoE): {((df['target_count'] > 1) & ~df['is_aoe']).sum()}")
print(f"\nNew columns added: target_count, is_aoe, aoe_type, aoe_size, estimated_targets")


=== Final Summary ===
Total spells: 574
With damage: 210
AoE: 161
Single-target: 316
Multi-target (non-AoE): 28

New columns added: target_count, is_aoe, aoe_type, aoe_size, estimated_targets


In [18]:
# Top AoE damage spells
print("\n=== Top 10 AoE Damage Spells ===")
top_aoe = damage_spells[damage_spells['is_aoe']].nlargest(10, 'avg_damage')
print(top_aoe[['spell_name', 'level', 'aoe_type', 'aoe_size', 'damage_dice', 'avg_damage']].to_string(index=False))


=== Top 10 AoE Damage Spells ===
                 spell_name  level aoe_type aoe_size damage_dice  avg_damage
               Disintegrate      6     cube  10-foot   10d6 + 40        75.0
               Meteor Swarm      9   radius  40-foot        20d6        70.0
                     Symbol      7   radius  60-foot       10d10        55.0
Abi-Dalzim's Horrid Wilting      8     cube  30-foot        12d8        54.0
           Incendiary Cloud      8   radius  20-foot        10d8        45.0
     Delayed Blast Fireball      7   radius  20-foot        12d6        42.0
                   Sunburst      8   radius  60-foot        12d6        42.0
                 Fire Storm      7     cube  10-foot        7d10        38.5
               Cone of Cold      5     cone  60-foot         8d8        36.0
             Conjure Volley      5   radius  40-foot         8d8        36.0


In [19]:
# Top single-target damage spells
print("\n=== Top 10 Single-Target Damage Spells ===")
top_single = damage_spells[damage_spells['target_count'] == 1].nlargest(10, 'avg_damage')
print(top_single[['spell_name', 'level', 'damage_dice', 'avg_damage']].to_string(index=False))


=== Top 10 Single-Target Damage Spells ===
                    spell_name  level damage_dice  avg_damage
                   Time Ravage      9       10d12        65.0
               Finger of Death      7    7d8 + 30        61.5
                          Harm      6        14d6        49.0
            Psychic Crush (UA)      6        12d6        42.0
                 Reality Break      8        6d12        39.0
                        Blight      4         8d8        36.0
Raulothim's Psychic Lance (UA)      4        10d6        35.0
                 Blade Barrier      6        6d10        33.0
         Negative Energy Flood      5        5d12        32.5
               Banishing Smite      5        5d10        27.5


# Save Updated Data

In [20]:
# Save the updated dataframe — strip duplicate .N suffix columns from prior re-runs
df_save = df[[c for c in df.columns if not re.match(r'.+\.\d+$', c)]]
df_save = df_save.loc[:, ~df_save.columns.duplicated()]
output_path = f"{DATA_DIR}/spells.csv"
df_save.to_csv(output_path, index=False)
print(f"Saved {len(df_save)} spells to {output_path}")
print(f"Columns ({len(df_save.columns)}): {list(df_save.columns)}")

Saved 574 spells to ./data/spells.csv
Columns (39): ['spell_name', 'source', 'level', 'school', 'casting_time', 'range', 'components', 'duration', 'description', 'level_scaling', 'spell_lists', 'damage_dice', 'damage_type', 'avg_damage', 'target_count', 'is_aoe', 'aoe_type', 'aoe_size', 'estimated_targets', 'base_dpr', 'effective_targets', 'modified_dpr', 'inflicts_blinded', 'inflicts_charmed', 'inflicts_deafened', 'inflicts_frightened', 'inflicts_incapacitated', 'inflicts_paralyzed', 'inflicts_petrified', 'inflicts_poisoned', 'inflicts_prone', 'inflicts_restrained', 'inflicts_stunned', 'grants_flying', 'ac_bonus', 'attack_bonus', 'save_bonus', 'grants_advantage', 'inflicts_disadvantage']


In [21]:
# Final summary
print("\n=== Final Summary ===")
print(f"Total spells: {len(df)}")
print(f"With damage: {(df['avg_damage'] > 0).sum()}")
print(f"AoE: {df['is_aoe'].sum()}")
print(f"Single-target: {(df['target_count'] == 1).sum()}")
print(f"Multi-target: {(df['target_count'] > 1).sum()}")


=== Final Summary ===
Total spells: 574
With damage: 210
AoE: 161
Single-target: 316
Multi-target: 28


# Merge Spellcasters

Determine which features a creature gains by having spells

In [22]:
# =============================================================================
# MERGE SPELLCASTERS — Round-Aware DPR + Buff Handling
# =============================================================================
# Load helper for creature melee DPR
if cwd.name == 'notebooks':
    from helper_files import parse_dpr_from_json, EXPECTED_COMBAT_ROUNDS
else:
    from notebooks.helper_files import parse_dpr_from_json, EXPECTED_COMBAT_ROUNDS

COMBAT_ROUNDS = EXPECTED_COMBAT_ROUNDS  # 3

# Prepare spell features for join (deduplicate columns from re-runs)
spell_feature_columns = ['spell_name', 'level', 'casting_time',
       'modified_dpr', 'inflicts_blinded',
       'inflicts_charmed', 'inflicts_deafened', 'inflicts_frightened',
       'inflicts_incapacitated', 'inflicts_paralyzed', 'inflicts_petrified',
       'inflicts_poisoned', 'inflicts_prone', 'inflicts_restrained',
       'inflicts_stunned', 'grants_flying', 'ac_bonus', 'attack_bonus',
       'save_bonus', 'grants_advantage', 'inflicts_disadvantage',
       ]
df_spells_joinable = df[spell_feature_columns].copy()
df_spells_joinable = df_spells_joinable.loc[:, ~df_spells_joinable.columns.duplicated()]

# Normalize spell names: lowercase, replace / with space (raw data inconsistency)
def _normalize_spell_name(s):
    return s.lower().replace('/', ' ')

df_spells_joinable['spell_name'] = df_spells_joinable['spell_name'].apply(_normalize_spell_name)

# Load spellcasters_spells.csv (now with frequency column)
df_spellcasters_spells = pd.read_csv(f"{DATA_DIR}/spellcasters_spells.csv")
df_spellcasters_spells['spell'] = df_spellcasters_spells['spell'].apply(_normalize_spell_name)
print(f"Loaded {len(df_spellcasters_spells)} creature-spell rows ({df_spellcasters_spells['Name'].nunique()} creatures)")
print(f"Columns: {list(df_spellcasters_spells.columns)}")

# Merge spell features onto creature-spell rows
expanded = df_spellcasters_spells.merge(
    df_spells_joinable,
    left_on='spell',
    right_on='spell_name',
    how='left'
).drop(columns=['spell_name'])

# Show unmatched spells BEFORE NaN fill (so we can detect them)
unmatched = expanded[expanded['level'].isna()]
if len(unmatched) > 0:
    print(f"\nUnmatched spells ({len(unmatched)}):")
    for _, row in unmatched.iterrows():
        print(f"  {row['Name']}: {row['spell']}")
else:
    print(f"\nAll {len(expanded)} creature-spell rows matched successfully")

# Fill NaN for spells not found in spells.csv (handles ALL column types)
for col in expanded.columns:
    if col in ('Name', 'spell', 'frequency', 'casting_time'):
        continue
    if expanded[col].dtype == 'bool':
        expanded[col] = expanded[col].fillna(False)
    elif expanded[col].dtype in ('int64', 'float64'):
        expanded[col] = expanded[col].fillna(0.0)
    elif expanded[col].dtype == 'object':
        # Boolean columns stored as object (e.g., from CSV True/False strings)
        try:
            expanded[col] = expanded[col].map(
                {'True': True, 'False': False, True: True, False: False}
            ).fillna(False).infer_objects(copy=False)
        except (TypeError, ValueError):
            expanded[col] = expanded[col].fillna(0)

# Load creature melee DPR from raw monster data
df_raw = pd.read_csv(f"{DATA_DIR}/dnd5e_monsters_from_json.csv")
creature_melee_dpr = dict(zip(df_raw['Name'], df_raw['Actions'].apply(parse_dpr_from_json)))

print(f"\nSample creature melee DPR:")
for name in ['Oni', 'Mage', 'Lich']:
    print(f"  {name}: {creature_melee_dpr.get(name, 0):.1f}")

Loaded 313 creature-spell rows (36 creatures)
Columns: ['Name', 'spell', 'frequency']

Unmatched spells (4):
  Glabrezu: power word stun
  Lich: acid arrow
  Lich: power word kill
  Lich: power word stun

Sample creature melee DPR:
  Oni: 30.0
  Mage: 4.5
  Lich: 10.5


/tmp/ipykernel_10808/1664492979.py:66: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(False).infer_objects(copy=False)
/tmp/ipykernel_10808/1664492979.py:66: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(False).infer_objects(copy=False)
/tmp/ipykernel_10808/1664492979.py:66: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  )

In [23]:
# =============================================================================
# BUFF SPELL RULES
# =============================================================================
BUFF_SPELL_RULES = {
    'shield':               {'spell_ac_bonus': 5, 'skip_dpr': True},    # reaction
    'mage armor':           {'spell_ac_bonus': 3, 'skip_dpr': True},    # pre-cast
    'shield of faith':      {'spell_ac_bonus': 2, 'skip_dpr': True},    # bonus action
    'barkskin':             {'spell_ac_bonus': 6, 'skip_dpr': True},    # pre-cast concentration
    'invisibility':         {'feature': 'spell_feature_invisibility', 'skip_dpr': True},
    'greater invisibility': {'feature': 'spell_feature_superior_invisibility',
                             'consumes_round': True, 'skip_dpr': True},
    'blur':                 {'consumes_round': True, 'skip_dpr': True},
    'fly':                  {'grants_flying': True, 'skip_dpr': True},
}

def _safe_bool(val):
    """Convert a value to bool, treating NaN/None as False."""
    if pd.isna(val):
        return False
    return bool(val)

def _safe_num(val):
    """Convert a value to float, treating NaN/None as 0."""
    if pd.isna(val):
        return 0.0
    return float(val)


def compute_creature_combat_features(creature_name, creature_spells, melee_dpr):
    """Compute round-aware combat features for a single creature."""
    result = {
        'Name': creature_name,
        'combat_dpr': 0.0,
        'spell_ac_bonus': 0,
        'spell_feature_invisibility': 0,
        'spell_feature_superior_invisibility': 0,
        'grants_flying': False,
        'attack_bonus': 0.0,
        'save_bonus': 0.0,
        'grants_advantage': False,
        'inflicts_disadvantage': False,
    }
    for cond in CONDITIONS:
        result[f'inflicts_{cond}'] = False
    
    available_attack_rounds = COMBAT_ROUNDS
    
    # --- Pass 1: Handle buff spells and aggregate non-DPR features ---
    for _, row in creature_spells.iterrows():
        spell_name = str(row.get('spell', '')).lower()
        casting_time = str(row.get('casting_time', '1 action')).lower()
        
        # Apply buff rules
        if spell_name in BUFF_SPELL_RULES:
            rules = BUFF_SPELL_RULES[spell_name]
            if 'spell_ac_bonus' in rules:
                result['spell_ac_bonus'] = max(result['spell_ac_bonus'], rules['spell_ac_bonus'])
            if 'feature' in rules:
                result[rules['feature']] = 1
            if 'grants_flying' in rules:
                result['grants_flying'] = True
            if rules.get('consumes_round', False):
                if 'bonus' not in casting_time and 'reaction' not in casting_time:
                    available_attack_rounds -= 1
        
        # Aggregate condition flags (MAX across all spells, NaN-safe)
        for cond in CONDITIONS:
            col = f'inflicts_{cond}'
            if col in row and _safe_bool(row[col]):
                result[col] = True
        
        # Aggregate other flags via MAX (NaN-safe)
        if _safe_bool(row.get('grants_advantage')):
            result['grants_advantage'] = True
        if _safe_bool(row.get('inflicts_disadvantage')):
            result['inflicts_disadvantage'] = True
        if _safe_bool(row.get('grants_flying')):
            result['grants_flying'] = True
        result['attack_bonus'] = max(result['attack_bonus'], _safe_num(row.get('attack_bonus')))
        result['save_bonus'] = max(result['save_bonus'], _safe_num(row.get('save_bonus')))
    
    available_attack_rounds = max(available_attack_rounds, 0)
    
    # --- Pass 2: Round-aware DPR allocation ---
    damage_options = []
    
    for _, row in creature_spells.iterrows():
        spell_name = str(row.get('spell', '')).lower()
        dpr = _safe_num(row.get('modified_dpr'))
        freq = str(row.get('frequency', 'at_will'))
        
        # Skip buff-only spells and zero-damage spells
        if spell_name in BUFF_SPELL_RULES and BUFF_SPELL_RULES[spell_name].get('skip_dpr', False):
            continue
        if dpr <= 0:
            continue
        
        # Determine uses per combat
        if freq == 'at_will':
            max_uses = COMBAT_ROUNDS
        elif freq.startswith('per_day_'):
            max_uses = int(freq.split('_')[-1])
        elif freq.startswith('slots_'):
            max_uses = int(freq.split('_')[-1])
        else:
            max_uses = 1
        
        damage_options.append({
            'name': spell_name,
            'dpr': dpr,
            'max_uses': min(max_uses, COMBAT_ROUNDS),
        })
    
    # Add melee as always-available fallback
    if melee_dpr > 0:
        damage_options.append({
            'name': '_melee_',
            'dpr': melee_dpr,
            'max_uses': COMBAT_ROUNDS,
        })
    
    # Sort by DPR descending for greedy allocation
    damage_options.sort(key=lambda x: x['dpr'], reverse=True)
    uses_remaining = {opt['name']: opt['max_uses'] for opt in damage_options}
    
    # Allocate damage to attack rounds (buff rounds consume first rounds)
    total_damage = 0.0
    for r in range(available_attack_rounds):
        for opt in damage_options:
            if uses_remaining[opt['name']] > 0:
                total_damage += opt['dpr']
                uses_remaining[opt['name']] -= 1
                break
    
    result['combat_dpr'] = total_damage / COMBAT_ROUNDS
    
    return result

print("compute_creature_combat_features() defined")

compute_creature_combat_features() defined


In [24]:
# =============================================================================
# APPLY ROUND-AWARE COMPUTATION TO ALL CREATURES
# =============================================================================
results = []
for creature_name, group in expanded.groupby('Name'):
    melee_dpr = creature_melee_dpr.get(creature_name, 0)
    result = compute_creature_combat_features(creature_name, group, melee_dpr)
    results.append(result)

df_spellcaster_features = pd.DataFrame(results)

# Verification: show key creatures
print("=== Round-Aware Combat DPR (key creatures) ===")
verify_creatures = ['Oni', 'Mage', 'Archmage', 'Lich', 'Cult Fanatic', 'Acolyte']
for name in verify_creatures:
    row = df_spellcaster_features[df_spellcaster_features['Name'] == name]
    if row.empty:
        continue
    r = row.iloc[0]
    melee = creature_melee_dpr.get(name, 0)
    extras = []
    if r['spell_ac_bonus'] > 0:
        extras.append(f"spell_ac_bonus={r['spell_ac_bonus']}")
    if r['spell_feature_invisibility']:
        extras.append("invis=1")
    if r['spell_feature_superior_invisibility']:
        extras.append("sup_invis=1")
    if r['grants_flying']:
        extras.append("flying=1")
    extra_str = f"  [{', '.join(extras)}]" if extras else ""
    print(f"  {name}: combat_dpr={r['combat_dpr']:.1f} (melee={melee:.1f}){extra_str}")

print(f"\nTotal creatures: {len(df_spellcaster_features)}")
print(f"Columns: {list(df_spellcaster_features.columns)}")

=== Round-Aware Combat DPR (key creatures) ===
  Oni: combat_dpr=68.0 (melee=30.0)  [invis=1, flying=1]
  Mage: combat_dpr=85.3 (melee=4.5)  [spell_ac_bonus=5, sup_invis=1, flying=1]
  Archmage: combat_dpr=144.0 (melee=4.5)  [spell_ac_bonus=3, invis=1, flying=1]
  Lich: combat_dpr=112.0 (melee=10.5)  [spell_ac_bonus=5, invis=1]
  Cult Fanatic: combat_dpr=16.5 (melee=9.0)  [spell_ac_bonus=2]
  Acolyte: combat_dpr=4.5 (melee=2.5)

Total creatures: 36
Columns: ['Name', 'combat_dpr', 'spell_ac_bonus', 'spell_feature_invisibility', 'spell_feature_superior_invisibility', 'grants_flying', 'attack_bonus', 'save_bonus', 'grants_advantage', 'inflicts_disadvantage', 'inflicts_blinded', 'inflicts_charmed', 'inflicts_deafened', 'inflicts_frightened', 'inflicts_incapacitated', 'inflicts_paralyzed', 'inflicts_petrified', 'inflicts_poisoned', 'inflicts_prone', 'inflicts_restrained', 'inflicts_stunned']


In [25]:
# Show full results table
display_cols = ['Name', 'combat_dpr', 'spell_ac_bonus', 'spell_feature_invisibility',
                'spell_feature_superior_invisibility', 'grants_flying', 
                'attack_bonus', 'save_bonus', 'grants_advantage', 'inflicts_disadvantage']
df_spellcaster_features[display_cols].sort_values('combat_dpr', ascending=False)

,Name,combat_dpr,spell_ac_bonus,spell_feature_invisibility,spell_feature_superior_invisibility,grants_flying,attack_bonus,save_bonus,grants_advantage,inflicts_disadvantage
2,Archmage,144.000000,3,1,0,True,0.0,0.0,True,False
27,Pit Fiend,112.000000,0,0,0,False,0.0,0.0,False,False
21,Lich,112.000000,5,1,0,False,0.0,0.0,False,False
31,Solar,98.000000,0,1,0,False,0.0,0.0,True,True
28,Planetar,87.333333,0,1,0,False,0.0,0.0,True,True
22,Mage,85.333333,5,0,1,True,0.0,0.0,False,False
32,Spirit Naga,84.000000,0,0,0,False,0.0,0.0,False,False
24,Mummy Lord,75.000000,2,0,0,False,0.0,0.0,True,True
26,Oni,68.000000,0,1,0,True,0.0,0.0,False,False
34,Storm Giant,60.000000,0,0,0,False,0.0,0.0,False,False


In [26]:
# Save to CSV
save_path = f"{DATA_DIR}/spellcaster_spell_features.csv"
df_spellcaster_features.to_csv(save_path, index=False)
print(f"Saved {len(df_spellcaster_features)} creatures to {save_path}")

Saved 36 creatures to ./data/spellcaster_spell_features.csv
